In [1]:
import matplotlib
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
import os
import numpy as np
from fairchem.core.datasets import AseDBDataset
from ase.calculators.emt import EMT 
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from ase.io.trajectory import Trajectory
from ase import units
from ase.io import read
from ase.db import connect
from ase.calculators.orca import ORCA, OrcaProfile

/opt/homebrew/Caskroom/miniconda/base/envs/SLAC/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-14 09:52:28,713	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
W0414 09:52:29.294000 8297 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
#dataset_path = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/train_4M" # using 4m dataset for exploration. Much smaller than the whole dataset
#dataset = AseDBDataset({"src": dataset_path})
#print(f"Total molecules in this split: {len(dataset)}")
data_dir = os.path.expanduser("/Users/joaopanicali/SLAC_v2/SLAC-atoms/train_4M")
shards = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".aselmdb")]
print(f"Found {len(shards)} shards")
db = connect(shards[0])

Found 80 shards


In [133]:
#query molecule with specific number of atoms
for row in db.select('natoms=2', limit=1):
    dataset_index = row.id - 1    
    print(f"Found {row.formula} natively at database ID {row.id} (Dataset index: {dataset_index})")
    atoms = row.toatoms()
    atoms.info["charge"] = row.data.get("charge", 0)
    atoms.info["spin"] = row.data.get("spin", 1)

Found H2 natively at database ID 5 (Dataset index: 4)


In [131]:
def parse_orca_hessian(filepath: str) -> np.ndarray:
    """Parses an ORCA .hess file and extracts the symmetric Hessian matrix."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    start_line = next((i + 2 for i, line in enumerate(lines) if "$hessian" in line), None)
    if start_line is None:
        raise ValueError("Hessian section not found in the .hess file.")

    dim = int(lines[start_line - 1].strip())
    hessian = np.zeros((dim, dim))
    i = start_line
    
    while i < len(lines):
        parts = lines[i].split()
        if not parts or "$" in lines[i]: 
            break
        if not any("." in p for p in parts):
            col_start = int(parts[0])
        else:
            row_idx = int(parts[0])
            for j, val in enumerate(map(float, parts[1:])):
                hessian[row_idx, col_start + j] = val
        i += 1

    return (hessian + hessian.T) / 2

In [ ]:
def compute_and_save_modes(atoms, orca_path: str, output_npz_path: str, scratch_dir: str = "./orca_scratch",
 freq_lb: float = 5.0, freq_ub: float = 1111.0):
    """
    Runs ORCA, extracts the Hessian, calculates normal modes using ASE Standard Units, 
    and saves the physics data to disk for rapid trajectory generation.
    """
    os.makedirs(scratch_dir, exist_ok=True)
    base_name = os.path.join(scratch_dir, "mol_opt_freq")
    
    atoms.calc = ORCA(
        profile=OrcaProfile(command=orca_path),
        directory="./orca_scratch",  
        label="mol_opt_freq",        
        orcasimpleinput='wB97M-D3BJ def2-TZVPD Opt Freq', #
        
        # change orca blocks for parallel processing
        orcablocks='%maxcore 4000\n%scf MaxIter 300 end',
        
        charge=atoms.info["charge"],
        mult=atoms.info["spin"], 
    )
    
    atoms.get_potential_energy() 
    
    R_0 = atoms.get_positions().flatten() 
    masses_amu = np.repeat(atoms.get_masses(), 3) 
    raw_hessian = parse_orca_hessian(f"/Users/joaopanicali/SLAC_v2/SLAC-atoms/orca_scratch/orca.hess")
    
    # ORCA outputs Hartree/Bohr^2. ASE converts this to eV/Angstrom^2
    conversion_factor = units.Hartree / (units.Bohr ** 2)
    hessian_ASE = raw_hessian * conversion_factor
    
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    tilde_H_ASE = M_inv_sqrt @ hessian_ASE @ M_inv_sqrt
    
    eigenvalues, V = np.linalg.eigh(tilde_H_ASE)

    
    # Take absolute value before sqrt to prevent NaN crashes from numerical noise (-1e-8)
    omegas_ASE = np.sqrt(np.abs(eigenvalues)) 

    raw_freqs_cm1 = omegas_ASE * 521.47

    #filtering frequencies of normal modes
    valid_indices = np.where(
        (raw_freqs_cm1 > freq_lb) & 
        (raw_freqs_cm1 < freq_ub)
    )[0]
    
    omegas_ASE_filtered = omegas_ASE[valid_indices]
    V_vib_filtered = V[:, valid_indices]

    
    np.savez_compressed(
        output_npz_path,
        R_0=R_0, #initial positions
        masses_amu=masses_amu, #molecule masses
        omegas_ASE=omegas_ASE_filtered, #mode frequencies (eigenvalues)
        V_vib=V_vib_filtered # normal modes (eigen vectors)
    )
    print(f"Physical mode data successfully saved to {output_npz_path}")

In [142]:
orca_command_path = "/Users/joaopanicali/software/orca_6/orca"
output_physics_file = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/mol_001_physics.npz"
compute_and_save_modes(
    atoms=atoms, 
    orca_path=orca_command_path, 
    output_npz_path=output_physics_file,
    scratch_dir="./orca_scratch"
)

              geometry optimization, check very carefully.
              ORCA does not by default supply the forces for the
              converged geometry!
Physical mode data successfully saved to /Users/joaopanicali/SLAC_v2/SLAC-atoms/mol_001_physics.npz


In [ ]:
def generate_dynamic_trajectory(npz_path: str, T_kelvin: float = 300.0, points_per_fast_cycle: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """
    This function generates trajectories. It determines the time delta to ensure that at least
    points_per_fast_cycle is sampled from highest frequency normal mode and the duration of the trajectory
    guarantees that we sample a full motion of the slowest mode.

    Outputs a time vector and coordinate matrix
    """
    
    data = np.load(npz_path)
    R_0 = data['R_0']
    masses_amu = data['masses_amu']
    omegas_ASE = data['omegas_ASE']
    V_vib = data['V_vib']
    
    num_modes = len(omegas_ASE)
    N_atoms = len(R_0) // 3
    
    # get periods in femptoseconds
    periods_fs = (2 * np.pi / omegas_ASE) / units.fs
    
    # get period of slowest vibration
    t_total_fs = np.max(periods_fs)  
    
    # define time step such that at least the defined number of points is sampled from the fastest normal mode's full motion
    dt_fs = np.min(periods_fs) / points_per_fast_cycle 
    
    time_grid_fs = np.arange(0, t_total_fs + dt_fs, dt_fs)
    num_frames = len(time_grid_fs)
    
    # Convert back to ASE units for the math engine
    time_steps_ASE = (time_grid_fs * units.fs).reshape(-1, 1) 
    
    print(f"--- Sampling Dynamics ---")
    print(f"Slowest Mode: {np.max(periods_fs):.2f} fs")
    print(f"Fastest Mode: {np.min(periods_fs):.2f} fs")
    print(f"Generated {num_frames} frames with dt = {dt_fs:.2f} fs")
    
    # get amplitude
    A_q = np.sqrt(2 * units.kB * T_kelvin) / omegas_ASE 
    
    # Random initial phases
    phi = np.random.uniform(0, 2 * np.pi, num_modes) 
    
    #calculating oscillaiton of mode
    c_t = A_q * np.cos(omegas_ASE * time_steps_ASE + phi) 
    
     
    # calculate mass-weighted displacements
    q_t = V_vib @ c_t.T 
    
    # convert back to cartesian coordinates
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    delta_R_Angstroms = M_inv_sqrt @ q_t 
    
    # extract final positions
    R_t = R_0[:, None] + delta_R_Angstroms 
    trajectory_tensor = R_t.T.reshape(num_frames, N_atoms, 3)
    
    return time_grid_fs, trajectory_tensor

In [146]:
generate_dynamic_trajectory(output_physics_file)

--- Sampling Dynamics ---
Slowest Mode: 2740.55 fs
Fastest Mode: 2725.31 fs
Generated 12 frames with dt = 272.53 fs


(array([   0.        ,  272.53118583,  545.06237166,  817.59355749,
        1090.12474332, 1362.65592915, 1635.18711498, 1907.71830081,
        2180.24948664, 2452.78067247, 2725.31185831, 2997.84304414]),
 array([[[ 7.56673668,  5.07078135,  0.56197003],
         [-7.56673668, -5.07078135, -0.53706062]],
 
        [[ 5.94600166,  2.21262688,  0.56065773],
         [-5.94600166, -2.21262688, -0.5383727 ]],
 
        [[ 2.05573042, -1.47854712,  0.55508353],
         [-2.05573042,  1.47854712, -0.5439465 ]],
 
        [[-2.62041154, -4.60980183,  0.54738464],
         [ 2.62041154,  4.60980183, -0.551645  ]],
 
        [[-6.29833658, -6.00024142,  0.54050894],
         [ 6.29833658,  6.00024142, -0.55852043]],
 
        [[-7.57422508, -5.12635527,  0.53708631],
         [ 7.57422508,  5.12635527, -0.56194303]],
 
        [[-5.96035035, -2.31911323,  0.53842275],
         [ 5.96035035,  2.31911323, -0.56060682]],
 
        [[-2.07151747,  1.36138619,  0.54400201],
         [ 2.07151747, 

In [ ]:
##ignore code below. 
def generate_trajectory(npz_path: str, num_frames: int = 1000, T_kelvin: float = 300.0, duration_ps: float = 1.0) -> np.ndarray:
    """
    Loads saved physical mode data and generates a synthetic 3D trajectory tensor
    using pure ASE Standard Units for flawless numerical stability.
    """
    # 1. Load Data
    data = np.load(npz_path)
    R_0 = data['R_0']
    masses_amu = data['masses_amu']
    omegas_ASE = data['omegas_ASE']
    V_vib = data['V_vib']
    
    num_modes = len(omegas_ASE)
    N_atoms = len(R_0) // 3
    
    # 2. Statistical Mechanics (Equipartition Theorem)
    # Total vibrational energy is k_B * T (1/2 potential, 1/2 kinetic).
    # E_total = 1/2 * omega^2 * A^2 = k_B * T --> A = sqrt(2 * k_B * T) / omega
    # Note: units.kB is exactly the Boltzmann constant in eV/K.
    A_q = np.sqrt(2 * units.kB * T_kelvin) / omegas_ASE 
    
    # 3. Vectorized Simulation 
    # Convert input duration (picoseconds) into femtoseconds, then into ASE time units.
    # units.fs represents the value of 1 femtosecond in ASE time.
    duration_fs = duration_ps * 1000
    time_steps_ASE = np.linspace(0, duration_fs * units.fs, num_frames).reshape(-1, 1) 
    
    # Random initial phases
    phi = np.random.uniform(0, 2 * np.pi, num_modes) 
    
    # Calculate modes over time. 
    # c_t shape: (num_frames, 3N-6)
    c_t = A_q * np.cos(omegas_ASE * time_steps_ASE + phi) 
    
    # Superposition in mass-weighted space. 
    # q_t shape: (3N, num_frames)
    q_t = V_vib @ c_t.T 
    
    # 4. Un-weight Masses 
    # Because q_t is in sqrt(amu * Angstrom^2), multiplying by 1/sqrt(amu) 
    # perfectly leaves delta_R naturally formatted in Angstroms!
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    delta_R_Angstroms = M_inv_sqrt @ q_t 
    
    # 5. Add to equilibrium geometry and format
    R_t = R_0[:, None] + delta_R_Angstroms 
    trajectory_tensor = R_t.T.reshape(num_frames, N_atoms, 3)
    
    return trajectory_tensor

In [ ]:
generate_trajectory(output_physics_file)